# PCR vs PLS: When Fewer Features Beat More
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/pcr_vs_pls.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import scale
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error

## Load and Prepare the Data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/Hitters.csv').dropna()
dummies = pd.get_dummies(df[['League', 'Division', 'NewLeague']])
y = df['Salary']
X = pd.concat([
    df.drop(['Salary', 'League', 'Division', 'NewLeague'], axis=1).astype('float64'),
    dummies[['League_N', 'Division_W', 'NewLeague_N']]
], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=1)
print(f"Dataset: {df.shape[0]} players, {X.shape[1]} features")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Target (Salary): mean=${y.mean():.0f}k, std=${y.std():.0f}k")

## Principal Component Analysis (PCA)

In [ ]:
pca = PCA()
X_train_pc = pca.fit_transform(scale(X_train))
X_test_pc = pca.transform(scale(X_test))

cum_var = np.cumsum(pca.explained_variance_ratio_) * 100
for k in [1, 2, 5, 7, 10, 15, 19]:
    print(f"  {k} components: {cum_var[k-1]:.1f}% variance explained")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
var_ratio = pca.explained_variance_ratio_ * 100
cum_var = np.cumsum(var_ratio)
ax.bar(range(1, 20), var_ratio, color='steelblue', alpha=0.7, label='Individual')
ax2 = ax.twinx()
ax2.plot(range(1, 20), cum_var, 'r-o', markersize=5, linewidth=2, label='Cumulative')
ax2.axhline(y=90, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Cumulative Variance Explained (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)', color='steelblue')
ax.tick_params(axis='y', labelcolor='steelblue')
ax.set_title('PCA: Variance Explained by Each Component')
ax.set_xticks(range(1, 20))
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

## PCA Loadings Heatmap

In [ ]:
loadings = pd.DataFrame(
    pca.components_[:5].T,
    columns=[f'PC{i+1}\n({pca.explained_variance_ratio_[i]*100:.1f}%)' for i in range(5)],
    index=X.columns
)
loadings = loadings.loc[loadings.iloc[:, 0].abs().sort_values(ascending=True).index]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(loadings.values, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(5))
ax.set_xticklabels(loadings.columns, fontsize=11)
ax.set_yticks(range(len(loadings)))
ax.set_yticklabels(loadings.index, fontsize=10)
ax.set_title('PCA Loadings: Feature Contributions to Top 5 Components')
for i in range(len(loadings)):
    for j in range(5):
        val = loadings.values[i, j]
        color = 'white' if abs(val) > 0.3 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)
plt.colorbar(im, ax=ax, label='Loading', shrink=0.8)
plt.tight_layout()
plt.show()

## PCR: Cross-Validation for Number of Components

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=1)
regr = LinearRegression()
pcr_mse = []
for k in range(1, X.shape[1] + 1):
    score = -cross_val_score(
        regr, X_train_pc[:, :k], y_train.to_numpy(),
        cv=kf, scoring='neg_mean_squared_error'
    ).mean()
    pcr_mse.append(score)

best_pcr_k = np.argmin(pcr_mse) + 1
print(f"Best PCR components (CV): {best_pcr_k}")
print(f"Best CV MSE: {pcr_mse[best_pcr_k - 1]:,.0f}")

## PLS: Cross-Validation

In [ ]:
pls_mse = []
for k in range(1, X.shape[1] + 1):
    pls = PLSRegression(n_components=k)
    score = -cross_val_score(
        pls, scale(X_train), y_train.to_numpy(),
        cv=kf, scoring='neg_mean_squared_error'
    ).mean()
    pls_mse.append(score)

best_pls_k = np.argmin(pls_mse) + 1
print(f"PLS CV minimum: {best_pls_k} components (MSE: {pls_mse[best_pls_k - 1]:,.0f})")
print(f"PLS with 2 components: MSE = {pls_mse[1]:,.0f}")
print("We select 2 for parsimony (nearly as good, much simpler)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(range(1, 20), pcr_mse, '-o', color='steelblue', markersize=5, linewidth=2)
ax1.axvline(x=best_pcr_k, color='red', linestyle='--', alpha=0.7, label=f'Best: {best_pcr_k} components')
ax1.set_xlabel('Number of Components')
ax1.set_ylabel('10-Fold CV MSE')
ax1.set_title('PCR: Cross-Validation MSE')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(1, 20, 2))

ax2.plot(range(1, 20), pls_mse, '-s', color='darkorange', markersize=5, linewidth=2)
ax2.axvline(x=2, color='green', linestyle='--', alpha=0.7, label='Selected: 2 (parsimonious)')
ax2.axvline(x=best_pls_k, color='red', linestyle=':', alpha=0.5, label=f'CV min: {best_pls_k}')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('10-Fold CV MSE')
ax2.set_title('PLS: Cross-Validation MSE')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(1, 20, 2))

ymin = min(min(pcr_mse), min(pls_mse)) * 0.92
ymax = max(max(pcr_mse), max(pls_mse)) * 1.05
ax1.set_ylim(ymin, ymax)
ax2.set_ylim(ymin, ymax)
plt.tight_layout()
plt.show()

## Test Set Evaluation

In [ ]:
# PCR with best_pcr_k components
regr_pcr = LinearRegression()
regr_pcr.fit(X_train_pc[:, :best_pcr_k], y_train)
pcr_test_mse = mean_squared_error(y_test, regr_pcr.predict(X_test_pc[:, :best_pcr_k]))

# Full OLS
regr_full = LinearRegression()
regr_full.fit(X_train_pc, y_train)
ols_test_mse = mean_squared_error(y_test, regr_full.predict(X_test_pc))

# PLS with 2 components
pls2 = PLSRegression(n_components=2)
pls2.fit(scale(X_train), y_train)
pls_test_mse = mean_squared_error(y_test, pls2.predict(scale(X_test)))

# Ridge
ridge = RidgeCV(alphas=np.logspace(-2, 6, 100), cv=kf)
ridge.fit(scale(X_train), y_train)
ridge_test_mse = mean_squared_error(y_test, ridge.predict(scale(X_test)))

print(f"Full OLS (19 features): MSE = {ols_test_mse:,.0f}, RMSE = ${np.sqrt(ols_test_mse):.0f}k")
print(f"PCR ({best_pcr_k} components):    MSE = {pcr_test_mse:,.0f}, RMSE = ${np.sqrt(pcr_test_mse):.0f}k")
print(f"PLS (2 components):     MSE = {pls_test_mse:,.0f}, RMSE = ${np.sqrt(pls_test_mse):.0f}k")
print(f"Ridge (alpha={ridge.alpha_:.1f}):  MSE = {ridge_test_mse:,.0f}, RMSE = ${np.sqrt(ridge_test_mse):.0f}k")

In [ ]:
methods = ['Full OLS\n(19 features)', f'PCR\n({best_pcr_k} components)', 'PLS\n(2 components)', 'Ridge']
mses = [ols_test_mse, pcr_test_mse, pls_test_mse, ridge_test_mse]
colors = ['#95a5a6', 'steelblue', 'darkorange', '#2ecc71']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, mses, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
for bar, mse in zip(bars, mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'{mse:,.0f}\n(RMSE: {np.sqrt(mse):.0f})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Test MSE')
ax.set_title('Test Set Performance: PCR vs PLS vs OLS vs Ridge')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(mses) * 1.2)
plt.tight_layout()
plt.show()

## Exercises

1. **Scree plot.** Plot the explained variance per component and the cumulative curve. How many components do you need to capture 95% of the variance?

2. **PLS loadings.** Compare the PLS weight vectors (`pls.x_weights_`) to the PCA loadings. Which features does PLS prioritise that PCA does not?

3. **Ridge vs PCR.** Add a Ridge regression (with `RidgeCV`) to the comparison. In what sense is Ridge a "soft" version of PCR?

4. **Log-transform the target.** Salary is right-skewed. Does predicting log(Salary) change which method wins?